# NLP05 — KoChatGPT 업그레이드
KoGPT2 기반 RLHF 3단계 파이프라인: SFT → RM → PPO

**루브릭**
- ① baseline vs SFT 비교
- ② SFT vs RM/PPO 비교
- ③ Decoding 실험 + LLM-as-a-Judge

In [ ]:
import os
os.chdir('/workspace/test/NLP/NLP05')
import sys
sys.path.insert(0, '/workspace/test/NLP/NLP05')
print('Working dir:', os.getcwd())

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'torch', 'nltk',
                'rouge-score', 'matplotlib', '--break-system-packages'])
import nltk
nltk.download('punkt', quiet=True)
print('설치 완료')

## STEP 1 — SFT (지도 미세조정)

### 학습 결과

| 항목 | 값 |
|------|-----|
| Best epoch | 3 |
| 소요 시간 | 239초 |
| cfg.epochs | 3 |
| cfg.batch_size | 8 |
| cfg.lr | 2e-05 |
| cfg.max_len | 128 |

**Epoch별 loss:**

| Epoch | train_loss | val_loss |
|-------|------------|----------|
| 1 | 2.1709 | - |
| 2 | 1.2735 | - |
| 3 | 1.1561 | - |


## STEP 2 — RM (보상 모델 학습)

### 학습 결과

| 항목 | 값 |
|------|-----|
| Best epoch | 3 |
| 소요 시간 | 278초 |
| cfg.epochs | 3 |
| cfg.batch_size | 8 |
| cfg.lr | 1e-05 |
| cfg.max_len | 128 |

**Epoch별 loss:**

| Epoch | train_loss | val_loss |
|-------|------------|----------|
| 1 | 0.7117 | - |
| 2 | 0.6906 | - |
| 3 | 0.6537 | - |


## STEP 3 — PPO (강화학습)

### 학습 결과

| 항목 | 값 |
|------|-----|
| Best epoch | 1 |
| 소요 시간 | 1103초 |
| cfg.epochs | 1 |
| cfg.batch_size | 4 |
| cfg.lr | 1e-05 |
| cfg.kl_coef | 0.1 |
| cfg.clip_eps | 0.2 |

**Epoch별 loss:**

| Epoch | train_loss | val_loss |
|-------|------------|----------|
| 1 | 172.8571 | - |


## STEP 4 — 모델 비교 평가

### ① baseline vs SFT 비교

| 프롬프트 | baseline | SFT |
|----------|----------|-----|
| 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡... | 미샤엘이 만든 화장품 브랜드인 '에스티로더'다. 에스티로더는 미샤엘이 만든 화장품 브랜드다. 에스티로더가 만... | '저는 인공지능 언어모델로써 답변을 생성하고 있으며, 현재 시점에서는 존재하지 않습니다. 따라서 해당 질문에... |
| 컴터 부팅하는데 시간이 오래걸려 | 요. 그런데 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게... | 요?'컴퓨터 부팅은 일반적으로 몇 분에서 몇 시간까지 걸립니다. 그러나 이는 컴퓨터 부팅에 대한 일반적인 규... |
| 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다... | " "그렇습니다." "그렇습니다. 그 대학은 그 대학보다 훨씬 더 규모가 큰 대학입니다." "그렇다면 그 대... | '저는 AI 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만 인터넷 검색 결과, 도나 게일 ... |
| 서너 대 밀려있어도 그런데 제가 백번 밀려있길래그 내 ... | 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그... | '그렇다면 그 사람이 다시 버스정류장에 가서 그 사람을 만나서 이야기를 나누고 그 사람이 내일 다시 버스정류... |
| 드라마 <아이리스>의 최고 시청률은 어떠했는가? | #20180720 #미롱_식단   | '<아이리스>의 최고 시청률은 드라마 <아이리스>의 평균 시청률인 약 12%입니다. |


### ② SFT vs PPO 비교

| 프롬프트 | SFT | PPO |
|----------|-----|-----|
| 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡... | '저는 인공지능 언어모델로써 답변을 생성하고 있으며, 현재 시점에서는 존재하지 않습니다. ... | '제가 AI 언어모델로 작성된 것이 아니라 실제 상황에서 언급된 것은 없습니다. 하지만 그... |
| 컴터 부팅하는데 시간이 오래걸려 | 요?'컴퓨터 부팅은 일반적으로 몇 분에서 몇 시간까지 걸립니다. 그러나 이는 컴퓨터 부팅에... | '제가 AI 모델은 아니지만 일반적으로 게임은 일반적으로 시간이 많이 걸릴 수 있습니다. ... |
| 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다... | '저는 AI 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만 인터넷 검색 ... | ' 그는 죽은 것으로 추정되며, 그녀의 친구이자 동료였던 그녀의 친구들은 그가 죽은 것이 ... |
| 서너 대 밀려있어도 그런데 제가 백번 밀려있길래그 내 ... | '그렇다면 그 사람이 다시 버스정류장에 가서 그 사람을 만나서 이야기를 나누고 그 사람이 ... | 그게 지금 그게 지금 그게 지금 그게 뭐라고' 그게 뭐라고나 그런게 그런게 있나. 그게 지... |
| 드라마 <아이리스>의 최고 시청률은 어떠했는가? | '<아이리스>의 최고 시청률은 드라마 <아이리스>의 평균 시청률인 약 12%입니다. | 드라마가 방영된 것은 2018년입니다. 배경은 드라마와 드라마의 장르에 따라 다르지만 일반... |


## STEP 5 — 정량 평가 (BLEU + ROUGE-L)

| 모델 | BLEU | ROUGE-L |
|------|------|---------|  
| baseline | 0.0184 | 0.0000 |
| SFT | 0.1742 | 0.0813 |
| PPO | 0.0631 | 0.0154 |

In [ ]:
import json, matplotlib.pyplot as plt
import config as cfg

with open(f'{cfg.RESULTS_DIR}/quant_summary.json') as f:
    quant = json.load(f)

stages = list(quant.keys())
bleu   = [quant[s].get('avg_bleu', quant[s].get('bleu', 0)) for s in stages]
rouge  = [quant[s].get('avg_rouge_l', quant[s].get('rouge_l', 0)) for s in stages]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(stages, bleu,  color=['#aec6cf','#779ecb','#4a7ebf'])
axes[0].set_title('BLEU Score')
axes[0].set_ylabel('BLEU')
axes[1].bar(stages, rouge, color=['#aec6cf','#779ecb','#4a7ebf'])
axes[1].set_title('ROUGE-L Score')
axes[1].set_ylabel('ROUGE-L')
plt.suptitle('baseline vs SFT vs PPO')
plt.tight_layout()
plt.savefig(f'{cfg.RESULTS_DIR}/quantitative_comparison.png', dpi=150)
plt.show()

## Decoding 실험 (PPO 모델)

### 샘플 1: 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡게 된 곳은?

| 전략 | 출력 |
|------|------|
| greedy | '제가 AI 언어모델로 작성된 것이 아니라 실제 상황에서 언급된 것은 없습니다. 하지만 그는 현재까지도 그의 아들인 카를로스 1세가 다스리고 있... |
| beam_2 | '이것은 정치적인 상황과 관련이 있을 수 있는 부분이 많아 정확한 답변을 제공할 수 없습니다. 하지만 그가 설립한 공장이 어디인지는 모르지만, ... |
| beam_4 | '제가 파악한 바에 의하면, 막스와 미샤엘이 설립한 공장에서 이후의 생산을 맡게 된 것은 아니라는 것이 밝혀졌음에도 불구하고, 그의 공장이 어디... |
| topk_10 | '바라샨 왕국의 수도인 파리에 집중되어있는 것과 관련된 것은 다음과 같은 이유로 인해 불안정하게 진행되었다 New Contra의 명성을 빌어 알... |
| topk_50 | 그리고 어느 지역을 말하는 마녀의 나라에서 그와 그들에게 있어서의 발전이 그들의 군대와 그들에게 그들을 그들은 하여금 이들에게 말하였다. 위원회... |
| topp_09 | 어둠의 여왕'이름이 검은색으로 바뀌는 묘사가 부족하여 이해하기 힘들다는 것으로 보임 『미래 전쟁], 문단에 언급되어 있기로는 이러한 것에 대해서... |


### 샘플 2: 컴터 부팅하는데 시간이 오래걸려

| 전략 | 출력 |
|------|------|
| greedy | '제가 AI 모델은 아니지만 일반적으로 게임은 일반적으로 시간이 많이 걸릴 수 있습니다. 하지만 게임을 즐기는 것이 가장 좋은 방법은 다음과 같... |
| beam_2 | '제한된 부분이 있으셨다면 그것은 "그렇다면 더 많은 시간이 걸릴 수도 있습니다. 하지만 그렇다고 해서 모든 것이 해결되는 것은 아닙니다" 라는... |
| beam_4 | 져있다는게 있나봐'그렇다면 이점은 다음과 같았으면 좋겠네요.\n그렇지 않다면 다른 방법이 있을 수 있겠지만 그것은 어려울 수 있겠지만 그렇다고 ... |
| topk_10 | 져 있죠'제가 AI 언어모델이므로 이 문제를 해결할 수 없으므로, 어떤 부분이 부족한지 알 수 있습니다. 다음은 자세한 내용이 없는 상태이시되는... |
| topk_50 | 생산성이 감소하고 있다. 새로운 기술이 반영된 부분은 기술의 발달로 생기는 것이므로 과거의 노력이 포함되지 않도록 시간이 좀 빨라졌기 때문'이라... |
| topp_09 | 요 이걸 반영하기 쉽코ia's a my sonation Fail re cannail y Topper twork late in Commera ra... |


### 샘플 3: 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다니던 대학교는?

| 전략 | 출력 |
|------|------|
| greedy | ' 그는 죽은 것으로 추정되며, 그녀의 친구이자 동료였던 그녀의 친구들은 그가 죽은 것이 아니라 그녀의 친구들과 함께 살았던 것으로 보인다. 멋... |
| beam_2 | 잃어버린 사람들의 모임에서 언급된 것은 "새로운 시작을 위한 새로운 시작을 위한 새로운 시작을 위한 새로운 시작을 알리는 것'이라는 제목의 뉴스... |
| beam_4 | 잃어버린 사람들의 모임에 의해 밝혀진 바에 의하면, 그의 어머니와 그의 동료들, 그리고 그의 동료들과 함께 살았던 사람들의 증언에 의하면, 그의... |
| topk_10 | 사건이 얽혀있는 곳은 미국의 정치판'사회가 그에게 영향을 주어 정치적인 영향력을 미치려 하지 말아달라고 덧붙였다면, 그녀의 죽음은 정치적인 상황... |
| topk_50 | '가족들이 죽은 날짜가 분명하지 않다고 여겨지는 상황에서 언급되고 있으나, "미국 플로리다 대학에서"라는 단어는 의미가 없다. 로레타 섬은 뉴펀... |
| topp_09 | 사건이 발생됨에 의하여 그녀가 살인사건으로 투옥되었으나 그의 생애와 그가 경험한 역사적인 사건들을 포함하고 있던 그녀의 아버지와 그들의 가족, ... |


## LLM-as-a-Judge (SFT vs PPO)

### 샘플 1

**[질문]** 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡게 된 곳은?

**[응답 A - SFT]** '저는 인공지능 언어모델로써 답변을 생성하고 있으며, 현재 시점에서는 존재하지 않습니다. 따라서 해당 질문에 대한 정확한 답변을 제공할 수 없습니다.

**[응답 B - PPO]** '제가 AI 언어모델로 작성된 것이 아니라 실제 상황에서 언급된 것은 없습니다. 하지만 그는 현재까지도 그의 아들인 카를로스 1세가 다스리고 있는 것으로 알려져 있습니다.적절한 표현을 쓰기 위해 정치적인 상황과 관련된 부분이 많이 영향을 미치면서 그의 아들인 미나카엘이 미국으로 이주하여 그의 공장을 운영하고 있는 것이 현재는 막스와 미샤엘이 사망한

**평가 기준:** 유창성 / 관련성 / 정보성 (각 1~5점)

### 샘플 2

**[질문]** 컴터 부팅하는데 시간이 오래걸려

**[응답 A - SFT]** 요?'컴퓨터 부팅은 일반적으로 몇 분에서 몇 시간까지 걸립니다. 그러나 이는 컴퓨터 부팅에 대한 일반적인 규칙과 절차에 따라 달라질 수 있습니다. 일반적으로 컴퓨터 부팅 시간은 일반적으로 몇 분에서 몇 시간까지 소요됩니다.

**[응답 B - PPO]** '제가 AI 모델은 아니지만 일반적으로 게임은 일반적으로 시간이 많이 걸릴 수 있습니다. 하지만 게임을 즐기는 것이 가장 좋은 방법은 다음과 같습니다.\n\n1. 게임을 즐기는 것이 가장 좋은 것은 게임의 특성상 게임을 즐기는 것이 가장 빠른 것이 아니라는 것을 인식해야합니다. 게임을 즐기는 것이 가장 중요한 것은 게임의 특성상 데이터에 대한 데이터

**평가 기준:** 유창성 / 관련성 / 정보성 (각 1~5점)

### 샘플 3

**[질문]** 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다니던 대학교는?

**[응답 A - SFT]** '저는 AI 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만 인터넷 검색 결과, 도나 게일 맨슨이 다니던 대학교는 미국 캘리포니아주 샌프란시스코에 위치한 캘리포니아 주립대학교입니다.

**[응답 B - PPO]** ' 그는 죽은 것으로 추정되며, 그녀의 친구이자 동료였던 그녀의 친구들은 그가 죽은 것이 아니라 그녀의 친구들과 함께 살았던 것으로 보인다. 멋진 것은 그가 죽은 것이 아니라 죽은 것이 아니라 죽은 것이 아니라 죽은 것이 아니라 다른 사람의 모습으로 변해 그녀의 나이로 세상을 떠났다.머스추세츠 대학의 법학대학 도서관에 따르면 그는 어떤 것을 ray New ...

**평가 기준:** 유창성 / 관련성 / 정보성 (각 1~5점)

## 결론 및 고찰

| 단계 | Best Loss | BLEU | ROUGE-L |
|------|-----------|------|---------|  
| SFT | 1.1561 | 0.1742 | 0.0813 |
| RM | 0.6537 | - | - |
| PPO | 172.8571 | 0.0631 | 0.0154 |

### RLHF 3단계 파이프라인 체험 요약

| 단계 | 핵심 학습 |
|------|----------|
| SFT | 형식과 말투를 학습시킬 수 있다 |
| RM | "좋은 답"을 점수로 정의할 수 있다 |
| PPO | RM이 허술하면 Reward Hacking이 발생한다 |

### 개선 방향
1. `tokenizer.padding_side = 'left'` 적용
2. KL penalty 계수 상향 (0.1 → 0.3~0.5)
3. RM 학습 데이터 품질 개선
4. PPO epochs 증가 또는 배치 크기 조정